# Single-cell baseline update

Update the neural baselines with matched main MLP blocks (three hidden layers, 512 units, SiLU) and latent dimension 128.

**Count Flow Map and Count-FM are read-only.** Their saved metrics, timings, and figures are reused. Missing core exports cause an error. This notebook has no core training or generation command. Linear and Sinkhorn results are also reused.

Run the cells in order in your original GPU environment. A second run reuses completed baseline fits, generated samples, timings, and metrics. Interrupted training resumes from its last validation checkpoint. scGen and scVIDR share one VAE training trajectory per seed and select their checkpoints separately.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/update_scrna_baselines.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the updated Count Flow Map project.")
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.runtime_env import conda_runtime_env

DATA = ROOT / "data/tahoe_panel/tahoe_condition_holdout.npz"
REFERENCE = ROOT / "outputs/scrna_paper_v2"
BASELINE_OUTPUT = ROOT / "outputs/scrna_baselines_matched"
OUTPUT = ROOT / "outputs/scrna_paper_v3"
CONFIG = ROOT / "configs/scrna_baselines_matched.json"
REPORT_ONLY = False  # True only after the revised baseline results are complete.

COMMAND = [sys.executable, "-u", str(ROOT / "scripts/update_scrna_baselines.py"),
           "--data", str(DATA), "--reference", str(REFERENCE),
           "--baseline-output", str(BASELINE_OUTPUT), "--output", str(OUTPUT),
           "--config", str(CONFIG)]

def run_update(*flags):
    command = COMMAND + list(flags)
    print("Running:", subprocess.list2cmdline(command), flush=True)
    with subprocess.Popen(command, cwd=ROOT, env=conda_runtime_env(),
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if code:
        raise RuntimeError(f"Update stopped with exit code {code}. Read the error above.")

print("Python:", sys.executable)
print("Existing core results:", REFERENCE)
print("New paper outputs:", OUTPUT)


## Check inputs

This reads the completed OT/non-OT exports, frozen evaluation rows, PCA, and any revised baseline caches. It performs no training, sampling, timing, or report writes. Keep `REFERENCE` pointing at the existing v2 results.


In [ ]:
run_update("--check-only")


## Train/resume revised baselines and assemble the report

The first run performs three seeds for NB-VAE, CPA, and the shared scGen/scVIDR VAE. Only missing baseline work runs. Validation uses the original fixed cells and PCA. The first full run must use the original GPU and PyTorch environment so new baseline timings are comparable with the cached count-model timings.

If the terminal command has already completed, this cell reuses its outputs. Set `REPORT_ONLY = True` to require a fully cached report rebuild.


In [ ]:
run_update(*(["--report-only"] if REPORT_ONLY else []))


## Updated table and architecture report

The table has the original eight count-model operating points, four revised neural baselines, and two cached non-neural baselines. Full LaTeX is in `paper_scrna_table.tex`, ready to paste into the manuscript. The CSV reports total parameters without claiming identical full-model parameter counts.


In [ ]:
import pandas as pd
from IPython.display import display, Image

display(pd.read_csv(OUTPUT / "paper_main_table.csv"))
display(pd.read_csv(OUTPUT / "architecture_and_parameters.csv"))
display(pd.read_csv(OUTPUT / "checkpoint_selection.csv"))


## Existing count-model figures

These files are copied unchanged from the final v2 export. No count-model sampling, timing, PCA fitting, or figure reconstruction occurs.


In [ ]:
display(Image(filename=str(OUTPUT / "figures/paper_scrna_main.png")))
display(Image(filename=str(OUTPUT / "figures/paper_scrna_appendix.png")))


## Manuscript wording after the revised runs finish

> We match the main MLP blocks across neural methods using three hidden layers of 512 units with SiLU activations. VAE-based baselines use 128 latent dimensions. All methods use the same data split and evaluation protocol.

The appendix should specify the matched blocks, retained auxiliary components, training budgets, and validation checkpoint selection. Use the new baseline numbers before updating comparative claims. Count Flow Map and Count-FM numbers remain unchanged.
